# ECA-LDNet reviewer evaluation

Evaluates three full-model seeds and twelve available Stage-4 ablation checkpoints. Synthetic paired inputs use 256×256. Real paired data keep native output dimensions and use 512×512 overlapping tiles (batch 1) to fit T4 memory.

Each dataset is processed and exported separately. Its `<dataset>_results.zip` contains model-specific folders, tables, all predictions, run manifests, training-history plots, and fixed-seed comparison grids.


In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir 'numpy==2.2.6' 'scipy==1.15.3' 'keras>=3.9,<4' pyiqa scikit-image fade-python
print('Pinned NumPy 2.2.6 and SciPy 1.15.3. Restart the Kaggle session now, then continue at the next cell.')


In [ ]:
from pathlib import Path
import json, os, shutil, sys, zipfile

INPUTS_ROOT = Path('/kaggle/input')
ZIP_HITS = list(INPUTS_ROOT.rglob('eca_ldnet_reviewer_test_package.zip'))
MANIFEST_HITS = list(INPUTS_ROOT.rglob('package_manifest.json'))
if len(ZIP_HITS) == 1:
    PACKAGE_ARCHIVE = ZIP_HITS[0]
    EXTRACT_ROOT = Path('/kaggle/working/eca_ldnet_reviewer_package')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PACKAGE_ARCHIVE) as zf:
        bad = zf.testzip()
        assert bad is None, f'Corrupt package entry: {bad}'
        root = EXTRACT_ROOT.resolve()
        for member in zf.infolist():
            target = (EXTRACT_ROOT / member.filename).resolve()
            assert target == root or root in target.parents, f'Unsafe ZIP path: {member.filename}'
        zf.extractall(EXTRACT_ROOT)
    PACKAGE_ROOT = EXTRACT_ROOT / 'reviewer_test_package'
elif len(MANIFEST_HITS) == 1:
    PACKAGE_ROOT = MANIFEST_HITS[0].parent
else:
    raise RuntimeError(f'Expected one uploaded package ZIP or extracted package manifest; ZIPs={ZIP_HITS}, manifests={MANIFEST_HITS}')

SOURCE_PACKAGE_ROOT = PACKAGE_ROOT
CODE_ROOT = PACKAGE_ROOT / 'code'
MODEL_MANIFEST = json.loads((PACKAGE_ROOT / 'package_manifest.json').read_text())
assert len(MODEL_MANIFEST['models']) == 15, MODEL_MANIFEST['counts']
sys.path.insert(0, str(CODE_ROOT))
os.environ['PYTHONPATH'] = str(CODE_ROOT)
print('Package:', PACKAGE_ROOT)
print('Models:', MODEL_MANIFEST['counts'])
for model in MODEL_MANIFEST['models']:
    checkpoint = PACKAGE_ROOT / model['checkpoint']
    assert checkpoint.is_file(), checkpoint
    print(model['model_id'], checkpoint.stat().st_size / 1e6, 'MB')

import tensorflow as tf
import keras
from packaging.version import Version
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
assert Version(keras.__version__) >= Version('3.9'), 'Install Keras >=3.9 before loading these checkpoints.'
print('GPU:', tf.config.list_physical_devices('GPU'))
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Enable a Kaggle GPU before evaluation.'
for gpu in GPUS:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

# Smoke-test every saved model and its dynamic-resolution rebuild before long evaluation.
import gc, numpy as np
from eca_ldnet.model import load_pretrained, build_eca_ldnet
VARIANT_FLAGS = {'full': {}, 'no_eca': {'use_eca': False},
                 'eca_only': {'use_eca': True, 'use_pa': False},
                 'no_attention': {'use_eca': False, 'use_pa': False},
                 'no_physics': {'use_physics': False}}
for spec in MODEL_MANIFEST['models']:
    fixed = load_pretrained(str(PACKAGE_ROOT / spec['checkpoint']), compile=False)
    has_physics = 'physics_output' in {layer.name for layer in fixed.layers}
    observed = len(fixed.get_layer('physics_output').input) == 4 if has_physics else True
    dynamic = build_eca_ldnet(img_size=None, physics_uses_observed_input=observed,
                              **VARIANT_FLAGS[spec['variant']])
    dynamic.set_weights(fixed.get_weights())
    for height, width in [(256, 256), (272, 304)]:
        output = dynamic(np.zeros((1, height, width, 3), np.float32), training=False)
        assert tuple(output.shape) == (1, height, width, 3), (spec['model_id'], output.shape)
    print('Reload + dynamic inference OK:', spec['model_id'], 'parameters:', dynamic.count_params())
    del fixed, dynamic, output
    tf.keras.backend.clear_session(); gc.collect()
print('All 15 checkpoints passed the pre-evaluation load/shape smoke test.')

In [ ]:
# Writable code copy plus 512x512 tiled inference for dynamic native-resolution inputs.
from pathlib import Path
import os, shutil, sys
if 'SOURCE_PACKAGE_ROOT' not in globals():
    SOURCE_PACKAGE_ROOT = PACKAGE_ROOT
PATCHED_PACKAGE_ROOT = Path('/kaggle/working/reviewer_test_package_patched')
PATCHED_PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE_PACKAGE_ROOT / 'code', PATCHED_PACKAGE_ROOT / 'code', dirs_exist_ok=True)
shutil.copy2(SOURCE_PACKAGE_ROOT / 'package_manifest.json', PATCHED_PACKAGE_ROOT / 'package_manifest.json')
for folder in ('models', 'metadata'):
    dest = PATCHED_PACKAGE_ROOT / folder
    if not dest.exists():
        dest.symlink_to(SOURCE_PACKAGE_ROOT / folder, target_is_directory=True)

evaluate_py = PATCHED_PACKAGE_ROOT / 'code' / 'eca_ldnet' / 'evaluate.py'
source = evaluate_py.read_text(encoding='utf-8')
start = source.index('def predict_full(model, img):')
end = source.index('\n\ndef evaluate_dataset', start)
tiled_predict = '''def _tile_starts(length, tile_size, overlap):
    if length <= tile_size:
        return [0]
    stride = tile_size - overlap
    count = int(np.ceil((length - tile_size) / stride)) + 1
    return np.linspace(0, length - tile_size, count).round().astype(int).tolist()


def _predict_tile_eager(model, tile):
    output = model(tile[np.newaxis], training=False)
    if hasattr(output, "numpy"):
        output = output.numpy()
    return np.asarray(output, dtype=np.float32)[0]


def predict_full(model, img, tile_size=512, overlap=96):
    # Preserve native H/W while limiting GPU activation memory to one tile.
    h, w = img.shape[:2]
    dynamic = any(dim is None for dim in model.input_shape[1:3])
    if not dynamic:
        pred = model.predict(img[np.newaxis], verbose=0)[0]
        return np.clip(pred[:h, :w], 0, 1).astype(np.float32)
    if tile_size % 16 or overlap < 0 or overlap >= tile_size // 2:
        raise ValueError("tile_size must be divisible by 16; overlap must be below half the tile")
    ys = _tile_starts(h, tile_size, overlap)
    xs = _tile_starts(w, tile_size, overlap)
    if len(ys) == 1 and len(xs) == 1:
        ph, pw = (-h) % 16, (-w) % 16
        tile = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode="reflect") if ph or pw else img
        return np.clip(_predict_tile_eager(model, tile)[:h, :w], 0, 1).astype(np.float32)
    accum = np.zeros((h, w, 3), np.float32)
    weight_sum = np.zeros((h, w, 1), np.float32)
    for iy, y0 in enumerate(ys):
        y1 = min(y0 + tile_size, h)
        for ix, x0 in enumerate(xs):
            x1 = min(x0 + tile_size, w)
            crop = img[y0:y1, x0:x1, :]
            th, tw = crop.shape[:2]
            ph, pw = (-th) % 16, (-tw) % 16
            tile = np.pad(crop, ((0, ph), (0, pw), (0, 0)), mode="reflect") if ph or pw else crop
            pred = _predict_tile_eager(model, tile)[:th, :tw]
            wy, wx = np.ones(th, np.float32), np.ones(tw, np.float32)
            if iy > 0:
                n = min(ys[iy-1] + tile_size - y0, th)
                wy[:n] = np.linspace(1/(n+1), 1, n, dtype=np.float32)
            if iy + 1 < len(ys):
                n = min(y1 - ys[iy+1], th)
                wy[-n:] = np.minimum(wy[-n:], np.linspace(1, 1/(n+1), n, dtype=np.float32))
            if ix > 0:
                n = min(xs[ix-1] + tile_size - x0, tw)
                wx[:n] = np.linspace(1/(n+1), 1, n, dtype=np.float32)
            if ix + 1 < len(xs):
                n = min(x1 - xs[ix+1], tw)
                wx[-n:] = np.minimum(wx[-n:], np.linspace(1, 1/(n+1), n, dtype=np.float32))
            weight = wy[:, None] * wx[None, :]
            accum[y0:y1, x0:x1] += pred * weight[..., None]
            weight_sum[y0:y1, x0:x1] += weight[..., None]
    return np.clip(accum / np.maximum(weight_sum, 1e-8), 0, 1).astype(np.float32)
'''
source = source[:start] + tiled_predict + source[end:]
evaluate_py.write_text(source, encoding='utf-8')

runner_py = PATCHED_PACKAGE_ROOT / 'code' / 'scripts' / 'run_reviewer_evaluation.py'
runner_source = runner_py.read_text(encoding='utf-8')
archive_block = '''    archive_base = output.parent / "reviewer_evaluation_results"
    archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=output))
    print(f"Evaluation complete. Results ZIP: {archive_path}")
'''
archive_skip_block = '''    if os.environ.get("REVIEWER_SKIP_COMBINED_ZIP") == "1":
        print(f"Evaluation complete. Dataset outputs: {output}")
    else:
        archive_base = output.parent / "reviewer_evaluation_results"
        archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=output))
        print(f"Evaluation complete. Results ZIP: {archive_path}")
'''
if archive_block in runner_source:
    runner_source = runner_source.replace(archive_block, archive_skip_block, 1)
elif archive_skip_block not in runner_source:
    raise RuntimeError(f"Could not locate combined-archive block in {runner_py}")
runner_py.write_text(runner_source, encoding='utf-8')
PACKAGE_ROOT = PATCHED_PACKAGE_ROOT
CODE_ROOT = PACKAGE_ROOT / 'code'
old_path = str(SOURCE_PACKAGE_ROOT / 'code')
while old_path in sys.path:
    sys.path.remove(old_path)
if str(CODE_ROOT) in sys.path:
    sys.path.remove(str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT))
os.environ['PYTHONPATH'] = str(CODE_ROOT)
print('Tiled inference enabled: 512x512 patches, overlap up to 96 px, batch size 1.')
print('Output dimensions remain native. Kaggle input weights are unchanged.')


In [ ]:
# Plot the saved four-stage training histories separately for every full/ablation model.
import csv, math
import matplotlib.pyplot as plt

HISTORY_ROOT = Path('/kaggle/working/reviewer_evaluation/training_history')
HISTORY_ROOT.mkdir(parents=True, exist_ok=True)
history_summary = []
for spec in MODEL_MANIFEST['models']:
    fig, axes = plt.subplots(4, 3, figsize=(15, 13), constrained_layout=True)
    fig.suptitle(f"{spec['model_id']} — saved Stage 1–4 training history")
    for stage in range(1, 5):
        if spec['family'] == 'full':
            history_path = PACKAGE_ROOT / 'metadata' / 'full' / f"seed{spec['seed']}" / f'stage{stage}' / 'history.csv'
        else:
            history_path = PACKAGE_ROOT / 'metadata' / 'ablations' / spec['variant'] / f"seed{spec['seed']}" / f'stage{stage}_history.csv'
        if not history_path.is_file():
            raise FileNotFoundError(f"Missing history for {spec['model_id']} stage {stage}: {history_path}")
        with history_path.open(newline='', encoding='utf-8-sig') as f:
            rows = list(csv.DictReader(f))
        if not rows:
            raise ValueError(f'Empty history: {history_path}')
        metrics = [('loss', 'val_loss', 'Loss'),
                   ('psnr_metric', 'val_psnr_metric', 'PSNR'),
                   ('ssim_metric', 'val_ssim_metric', 'SSIM')]
        for col, (train_key, val_key, title) in zip(axes[stage - 1], metrics):
            epochs = [int(float(r.get('epoch', i + 1))) for i, r in enumerate(rows)]
            if train_key not in rows[0] or val_key not in rows[0]:
                col.set_title(f'Stage {stage}: {title} unavailable')
                col.axis('off')
                continue
            train = [float(r[train_key]) if r.get(train_key) else float('nan') for r in rows]
            val = [float(r[val_key]) if r.get(val_key) else float('nan') for r in rows]
            col.plot(epochs, train, label='train')
            col.plot(epochs, val, label='validation')
            col.set_title(f'Stage {stage}: {title} ({len(rows)} epochs)')
            col.set_xlabel('Epoch')
            col.grid(alpha=.25)
            col.legend()
        val_key = 'val_psnr_metric'
        best = max(((float(r[val_key]), int(float(r.get('epoch', i + 1))))
                    for i, r in enumerate(rows) if r.get(val_key)), default=(float('nan'), -1))
        history_summary.append({'model_id': spec['model_id'], 'variant': spec['variant'],
                                'seed': spec['seed'], 'stage': stage, 'epochs_recorded': len(rows),
                                'best_val_psnr': best[0], 'best_val_psnr_epoch': best[1],
                                'history_file': str(history_path)})
    model_plot_dir = HISTORY_ROOT / spec['model_id']
    model_plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(model_plot_dir / 'training_history_stages1-4.png', dpi=160)
    plt.close(fig)

summary_path = HISTORY_ROOT / 'training_history_summary.csv'
with summary_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(history_summary[0]))
    writer.writeheader()
    writer.writerows(history_summary)
print('Saved 15 per-model history plots and summary:', HISTORY_ROOT)


## Attach and map datasets

The two paths below are the Kaggle mounts already used in the training/evaluation runbooks. RESIDE-6K is added only when a separate hazy/GT test split exists; training folders are never used as test data. Edit `REAL_PAIRED_PATHS` and `RTTS_HAZY` to match the datasets you attach. The official source links and dataset names are in `REVIEWER_TEST_PACKAGE_README.md`.

In [ ]:
from pathlib import Path
import json

RESIDE_TEST_ROOT = Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/test')
SOTS_ROOT = Path('/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside')

def image_dir_pair(hazy, gt):
    return Path(hazy).is_dir() and Path(gt).is_dir()

PAIRED_DATASETS = []
# Only use a distinct held-out test split, never the training images.
reside_hazy, reside_gt = RESIDE_TEST_ROOT / 'hazy', RESIDE_TEST_ROOT / 'GT'
if image_dir_pair(reside_hazy, reside_gt):
    PAIRED_DATASETS.append({'name': 'RESIDE-6K-Test', 'hazy_dir': str(reside_hazy), 'gt_dir': str(reside_gt), 'size': 256})
else:
    print('RESIDE-6K test pair folders absent; skipping.')

for label, relative in [('SOTS-Indoor', 'indoor'), ('SOTS-Outdoor', 'outdoor')]:
    hazy, gt = SOTS_ROOT / relative / 'hazy', SOTS_ROOT / relative / 'clear'
    if image_dir_pair(hazy, gt):
        PAIRED_DATASETS.append({'name': label, 'hazy_dir': str(hazy), 'gt_dir': str(gt), 'size': 256})
    else:
        print('SOTS folders absent; fix SOTS_ROOT/layout or skip:', hazy, gt)

REAL_PAIRED_PATHS = {
    'O-HAZE':   (Path('/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY/hazy'), Path('/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY/GT')),
    'I-HAZE':   (Path('/kaggle/input/datasets/litheshshetty/i-haze/I-HAZE/test/hazy'), Path('/kaggle/input/datasets/litheshshetty/i-haze/I-HAZE/test/clear')),
    'NH-HAZE':  (Path('/kaggle/input/datasets/litheshshetty/nh-haze/NH-HAZE/test/hazy'), Path('/kaggle/input/datasets/litheshshetty/nh-haze/NH-HAZE/test/clear')),
    'DNH-HAZE': (None, None),
}
for name, paths in REAL_PAIRED_PATHS.items():
    if paths[0] is None or paths[1] is None:
        print('Not attached/configured; skip:', name)
        continue
    hazy, gt = map(Path, paths)
    if image_dir_pair(hazy, gt):
        PAIRED_DATASETS.append({'name': name, 'hazy_dir': str(hazy), 'gt_dir': str(gt), 'native_resolution': True})
    else:
        print('Invalid real paired paths; correct these before running:', name, hazy, gt)

# RTTS is unpaired. The evaluator reads images directly inside the selected folder.
RTTS_ROOT = Path('/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS')
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
image_folders = []
if RTTS_ROOT.is_dir():
    for folder in [RTTS_ROOT, *(p for p in RTTS_ROOT.rglob('*') if p.is_dir())]:
        count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        if count:
            image_folders.append((folder, count))
    print('RTTS folders containing images directly:')
    for folder, count in sorted(image_folders, key=lambda x: (len(x[0].parts), str(x[0]))):
        print(f'  {folder}: {count} images')
else:
    print('RTTS root not found:', RTTS_ROOT)

preferred = [x for x in image_folders if x[0].name.lower() in {'jpegimages', 'images', 'hazy'}]
if len(preferred) == 1:
    RTTS_HAZY = preferred[0][0]
elif not preferred and len(image_folders) == 1:
    RTTS_HAZY = image_folders[0][0]
else:
    RTTS_HAZY = None
    print('RTTS folder is ambiguous; after reviewing the printed counts, set RTTS_HAZY to the image-containing folder.')

UNPAIRED_DATASETS = []
if RTTS_HAZY is not None and Path(RTTS_HAZY).is_dir():
    UNPAIRED_DATASETS.append({'name': 'RTTS', 'hazy_dir': str(Path(RTTS_HAZY))})

assert PAIRED_DATASETS or UNPAIRED_DATASETS, 'No evaluation dataset folders found; check the mounted Kaggle paths.'
DATASET_CONFIG = Path('/kaggle/working/reviewer_dataset_config.json')
DATASET_CONFIG.write_text(json.dumps({'paired': PAIRED_DATASETS, 'unpaired': UNPAIRED_DATASETS}, indent=2))
print('Paired datasets:', json.dumps(PAIRED_DATASETS, indent=2))
print('Unpaired datasets:', json.dumps(UNPAIRED_DATASETS, indent=2))


## Optional validation-only blend and epsilon sweep

This uses the packaged seed 17 validation split and the six blend / three epsilon values requested by the reviewers. It runs only when the original ITS and RESIDE training inputs are attached at the paths recorded in the split manifest. It never reads SOTS or the real test sets for parameter selection. The sweep reports PSNR/SSIM and bootstrap intervals; it is an inference screening study, not a retrained learnable-blend comparison.

In [ ]:
import csv, subprocess, sys

full17 = next(m for m in MODEL_MANIFEST['models'] if m['model_id'] == 'full_seed17')
SPLIT_MANIFEST = PACKAGE_ROOT / full17['split_manifest']
with SPLIT_MANIFEST.open(newline='', encoding='utf-8-sig') as f:
    validation_rows = [r for r in csv.DictReader(f) if r.get('split') == 'validation']
from eca_ldnet.data import clean_scene_id, get_gt_path
TRAIN_DIR_PAIRS = [(Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/hazy'),
                    Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/GT')),
                   (Path('/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/hazy'),
                    Path('/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/clear'))]
sweep_pairs, missing_validation_paths = [], []
for row in validation_rows:
    found = None
    for hazy_dir, gt_dir in TRAIN_DIR_PAIRS:
        candidate = hazy_dir / Path(row['hazy_path']).name
        target = get_gt_path(candidate.name, str(gt_dir)) if candidate.is_file() and gt_dir.is_dir() else None
        if target and clean_scene_id(target) == row['clean_sha256']:
            found = (str(candidate), target); break
    if found: sweep_pairs.append({'hazy_path': found[0], 'clean_path': found[1], 'clean_sha256': row['clean_sha256'], 'split': 'validation'})
    else: missing_validation_paths.append(row['hazy_path'])
RELOCATED_VALIDATION = Path('/kaggle/working/reviewer_validation_seed17.csv')
if sweep_pairs:
    with RELOCATED_VALIDATION.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['hazy_path','clean_path','clean_sha256','split']); writer.writeheader(); writer.writerows(sweep_pairs)
RUN_VALIDATION_BLEND_SWEEP = bool(validation_rows) and not missing_validation_paths
if RUN_VALIDATION_BLEND_SWEEP:
    sweep_out = Path('/kaggle/working/reviewer_evaluation/validation_sweeps/seed17_blend_epsilon')
    sweep_cmd = [sys.executable, str(CODE_ROOT / 'scripts/sweep_asm_blend.py'),
                 '--model', str(PACKAGE_ROOT / full17['checkpoint']),
                 '--manifest', str(RELOCATED_VALIDATION), '--split', 'validation',
                 '--preprocessing', 'radiometric', '--output', str(sweep_out),
                 '--weights', '0', '0.02', '0.05', '0.08', '0.10', '0.20',
                 '--epsilons', '0.05', '0.10', '0.20', '--bootstrap-samples', '5000', '--seed', '42']
    subprocess.run(sweep_cmd, check=True)
else:
    print('Skipped validation sweep: attach ITS and RESIDE-6K training inputs or correct TRAIN_DIR_PAIRS. Matched validation pairs:', len(sweep_pairs), '/', len(validation_rows), 'missing:', len(missing_validation_paths))

## Run each dataset and save its evidence separately

The driver runs one dataset at a time across all 15 checkpoints. Progress is retained in `progress.json`; rerunning the cell skips completed model/dataset tasks. Each fully completed dataset is immediately exported as its own `<dataset>_results.zip`, with per-model tables, run manifests, all predictions, history plots, and fixed-seed visual comparison grids. Large native-resolution real images use overlapping 512×512 tiles at batch size 1. The optional high-resolution benchmark is disabled during this run.


In [ ]:
# Build one portable evidence folder/ZIP per completed dataset.
import csv, json, math, random, re, shutil, zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from eca_ldnet.data import get_gt_path

DATASET_EXPORT_ROOT = Path('/kaggle/working/dataset_results')
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
def _slug(text):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', text).strip('_')
def _rgb_thumb(path, max_side=360):
    im=Image.open(path).convert('RGB'); im.thumbnail((max_side,max_side),Image.Resampling.LANCZOS)
    return np.asarray(im)
def _make_grids(dataset, bundle):
    name=dataset['name']; safe=_slug(name); paired='gt_dir' in dataset
    grid=bundle/'evidence'/'grids'; grid.mkdir(parents=True,exist_ok=True)
    specs=MODEL_MANIFEST['models']; rng=random.Random(42)
    if paired:
        hazy_dir,gt_dir=Path(dataset['hazy_dir']),Path(dataset['gt_dir'])
        files=sorted(p for p in hazy_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
        samples=[(p,get_gt_path(p.name,str(gt_dir))) for p in files]
        samples=[(p,g) for p,g in samples if g]
    else:
        hazy_dir=Path(dataset['hazy_dir'])
        samples=sorted(p for p in hazy_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    selected=sorted(rng.sample(samples,min(4,len(samples))),key=lambda x:(x[0].name if paired else x.name))
    names=[(a.name if paired else a.name) for a in selected]
    (grid/'selected_examples.json').write_text(json.dumps({'sampling':'up to four uniformly sampled examples; fixed seed 42; not metric-ranked','images':names},indent=2))
    for sample in selected:
        hazy,gt=sample if paired else (sample,None)
        panels=[('Hazy',hazy)]
        if paired: panels.append(('Ground truth',Path(gt)))
        for spec in specs:
            pred=RESULTS_ROOT/('paired' if paired else 'unpaired')/spec['model_id']/safe/f'tested_sample_{safe}'/f'{hazy.stem}.png'
            if pred.is_file(): panels.append((spec['model_id'],pred))
        cols=5 if paired else 4; rows=math.ceil(len(panels)/cols)
        fig,axes=plt.subplots(rows,cols,figsize=(20 if paired else 18,4.2*rows),squeeze=False)
        fig.suptitle(f'{name} | {hazy.name} | checkpoint comparison',fontsize=14)
        for ax,(label,path) in zip(axes.flat,panels):
            ax.imshow(_rgb_thumb(path)); ax.set_title(label,fontsize=8); ax.axis('off')
        for ax in axes.flat[len(panels):]: ax.axis('off')
        fig.tight_layout()
        fig.savefig(grid/f'{hazy.stem}_comparison_grid.jpg',dpi=130,bbox_inches='tight'); plt.close(fig)
def export_dataset_bundle(dataset):
    name=dataset['name']; safe=_slug(name); paired='gt_dir' in dataset
    bundle=DATASET_EXPORT_ROOT/f'{safe}_results'; bundle.mkdir(parents=True,exist_ok=True)
    # Remove any previous copied model tree: the ZIP streams model outputs
    # directly from RESULTS_ROOT so predictions are not duplicated on disk.
    copied_models=bundle/'models'
    if copied_models.exists(): shutil.rmtree(copied_models)
    _make_grids(dataset,bundle)
    tables=bundle/'tables'; tables.mkdir(parents=True,exist_ok=True)
    for fn in ('per_model_dataset_summary.csv','across_seed_summary.csv'):
        src=RESULTS_ROOT/fn
        if src.is_file():
            with src.open(newline='',encoding='utf-8-sig') as f: rows=list(csv.DictReader(f))
            rows=[r for r in rows if r.get('dataset')==name]
            if rows:
                with (tables/fn).open('w',newline='',encoding='utf-8') as f:
                    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    if HISTORY_ROOT.is_dir(): shutil.copytree(HISTORY_ROOT,bundle/'training_history',dirs_exist_ok=True)
    protocol=('native-size images via 512x512 tiles, batch 1, overlap-weighted blending' if dataset.get('native_resolution') else '256x256 paired synthetic protocol' if paired else 'native-size unpaired protocol')
    (bundle/'README.txt').write_text(f'{name} reviewer evidence\nProtocol: {protocol}\nContains per-checkpoint manifests, CSV tables, all generated predictions, training-history plots, and up to four fixed-seed visual comparison grids. The grids are sampled without looking at metrics. RTTS no-reference scores are perceptual proxies, not ground-truth fidelity.\n',encoding='utf-8')
    archive=bundle.with_suffix('.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as zf:
        # Include evidence files generated inside the bundle.
        for path in sorted(bundle.rglob('*')):
            if path.is_file() and copied_models not in path.parents:
                zf.write(path,arcname=f'{bundle.name}/{path.relative_to(bundle).as_posix()}')
        # Stream each model's CSVs, manifests, and predictions into the ZIP.
        for spec in MODEL_MANIFEST['models']:
            src=RESULTS_ROOT/('paired' if paired else 'unpaired')/spec['model_id']/safe
            if src.is_dir():
                for path in sorted(src.rglob('*')):
                    if path.is_file():
                        arc=f'{bundle.name}/models/{spec["model_id"]}/{path.relative_to(src).as_posix()}'
                        zf.write(path,arcname=arc)
    print(f'Dataset bundle: {bundle}\nZIP: {archive} ({archive.stat().st_size/1e9:.2f} GB)')


In [ ]:
import subprocess, sys
RESULTS_ROOT=Path('/kaggle/working/reviewer_evaluation')
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
failures=[]
for dataset in PAIRED_DATASETS+UNPAIRED_DATASETS:
    name=dataset['name']
    cmd=[sys.executable,str(CODE_ROOT/'scripts/run_reviewer_evaluation.py'),
         '--package-root',str(PACKAGE_ROOT),'--dataset-config',str(DATASET_CONFIG),
         '--output',str(RESULTS_ROOT),'--perceptual','lpips','dists',
         '--nr-metrics','niqe','brisque','musiq','maniqa','clipiqa',
         '--bootstrap-samples','5000','--seed','42','--resume','--dataset-names',name]
    if UNPAIRED_DATASETS and 'gt_dir' not in dataset: cmd.append('--fade')
    print(f'Running {name}: {len(MODEL_MANIFEST["models"])} checkpoints')
    run_env=os.environ.copy()
    run_env['REVIEWER_SKIP_COMBINED_ZIP']='1'
    run=subprocess.run(cmd,check=False,env=run_env)
    if run.returncode==0:
        export_dataset_bundle(dataset)
    else:
        failures.append(name)
        print(f'{name} failed; subsequent datasets will continue. Fix the printed issue, then rerun this cell; --resume skips completed model/dataset tasks.')
print('Dataset ZIPs:',*[str(p) for p in sorted(DATASET_EXPORT_ROOT.glob('*_results.zip'))],sep='\n  ')
if failures: print('Datasets still needing a successful rerun:',failures)
else: print('All enabled datasets completed and each has its own evidence ZIP.')
print('All task outputs remain organized under:',RESULTS_ROOT)


In [ ]:
# Optional deployment benchmark. Leave off during full dataset inference; it has a separate non-tiled high-resolution path.
RUN_RESOLUTION_BENCHMARK = False
if RUN_RESOLUTION_BENCHMARK:
    benchmark_cmd = [sys.executable, str(CODE_ROOT / 'scripts/benchmark_reviewer_models.py'),
                     '--package-root', str(PACKAGE_ROOT),
                     '--output', str(RESULTS_ROOT / 'resolution_latency_memory.csv'),
                     '--resolutions', '256x256', '512x512', '1280x720', '1920x1080',
                     '--warmup', '10', '--runs', '50']
    subprocess.run(benchmark_cmd, check=True)
    # Refresh the downloadable archive to include the benchmark CSV/JSON.
    shutil.make_archive('/kaggle/working/reviewer_evaluation_results', 'zip', root_dir=RESULTS_ROOT)
    print('Updated results ZIP with deployment benchmark.')
else:
    print('Skipped resolution benchmark; no latency/memory evidence was added.')

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
archives = sorted(Path('/kaggle/working/dataset_results').glob('*_results.zip'))
if not archives:
    print('No complete dataset ZIPs yet. Check the evaluation cell for failed datasets.')
for archive in archives:
    print(archive.name, f'{archive.stat().st_size / 1e9:.2f} GB')
    display(FileLink(str(archive)))
print('Per-dataset folders:', Path('/kaggle/working/dataset_results'))
